# Customer Support Agent — Evaluation, LangSmith Observability & OpenTelemetry

**Goal:** Build a small e-commerce customer support agent with **LangGraph**, trace every run in **LangSmith**, and learn how to evaluate it like a real ML system.

By the end of this notebook you will have:
1. A LangGraph agent that classifies support tickets into clear categories
2. A synthetic ticket dataset with ground-truth labels
3. LangSmith traces for every prediction (clickable in the UI), emitted through OpenTelemetry
4. Accuracy / precision / recall metrics
5. A CSV of results you can open in a spreadsheet and annotate
6. A simple loop: **inspect failures → tweak the prompt → re-run → compare**

---

## The workflow you'll follow

1. **Run the agent** on the synthetic tickets.
2. **Look at the metrics** and open the exported CSV in Google Sheets / Excel.
3. **Add validator comments** in the `validator_comment` column — flag anything that looks wrong, ambiguous, or surprising.
4. **Cluster the failures** into 2–4 *failure categories* (e.g., "misroutes complaints as questions", "confuses refund vs. return").
5. **Pick the one failure category that matters most** for your use case.
6. **Tweak the classifier prompt** at the bottom of the notebook to address it.
7. **Re-run** the agent and compare the new metrics + LangSmith/OpenTelemetry traces.

## 1. Install dependencies

This notebook calls Nebius Token Factory through LangChain. You will need a Nebius Token Factory API key and access to the configured model.

In [1]:
%pip install --quiet langgraph "langsmith[otel]>=0.4.25" langchain-nebius langchain-core pandas scikit-learn pydantic tqdm opentelemetry-sdk opentelemetry-exporter-otlp


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set up Nebius and LangSmith credentials

The notebook uses Nebius Token Factory for ticket generation and classification. It prompts for **`NEBIUS_API_KEY`** using a hidden input if the key is not already set in the environment. The key is used in memory for Nebius requests and is not written into this notebook.

The default endpoint is `https://api.tokenfactory.nebius.com/v1/` and the default model is `Qwen/Qwen3-30B-A3B-Instruct-2507`. The setup cell checks that the configured model is available to your API key before continuing. Set `NEBIUS_BASE_URL` or `NEBIUS_MODEL` before starting Jupyter to override them.

The notebook also prompts for **`LANGSMITH_API_KEY`** using a hidden input if it is not already set. That must be a LangSmith key (it is separate from the Nebius inference key). If LangSmith returns 403, check the key type/permissions and set `LANGSMITH_ENDPOINT` to the API region where the key was created (US default: `https://api.smith.langchain.com`; EU: `https://eu.api.smith.langchain.com`; APAC: `https://apac.api.smith.langchain.com`).


In [2]:
import os
import getpass

# Clear stale tracing variables so an old endpoint does not make OTel export to a 404 URL.
for k in [
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_TRACING_V2",
    "OTEL_EXPORTER_OTLP_ENDPOINT",
    "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT",
    "OTEL_EXPORTER_OTLP_HEADERS",
    "OTEL_EXPORTER_OTLP_TRACES_HEADERS",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
]:
    os.environ.pop(k, None)

def _prompt_for_secret(env_name: str, label: str) -> None:
    """Allow replacing stale notebook credentials without displaying them."""
    current_value = os.environ.get(env_name, "")
    entered_value = getpass.getpass(
        f"{label} (press Enter to keep the current value): "
    ).strip()
    if entered_value:
        os.environ[env_name] = entered_value
    elif not current_value:
        raise ValueError(f"{env_name} is required to run this notebook.")

_prompt_for_secret("NEBIUS_API_KEY", "Nebius Token Factory API key")
_prompt_for_secret("LANGSMITH_API_KEY", "LangSmith API key")

# Personal LangSmith tokens use their account's default workspace. Remove a stale
# workspace override inherited from a previous notebook or shell session.
if os.environ["LANGSMITH_API_KEY"].startswith("lsv2_pt_"):
    os.environ.pop("LANGSMITH_WORKSPACE_ID", None)

NEBIUS_BASE_URL = os.environ.get(
    "NEBIUS_BASE_URL", "https://api.tokenfactory.nebius.com/v1/"
)
NEBIUS_MODEL = os.environ.get(
    "NEBIUS_MODEL", "Qwen/Qwen3-30B-A3B-Instruct-2507"
)

# Confirm that this API key can access the selected model before running the notebook.
# If the model is unavailable, show the Qwen model IDs this key can access.
from openai import OpenAI

nebius_client = OpenAI(
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url=NEBIUS_BASE_URL,
)
available_model_ids = {model.id for model in nebius_client.models.list().data}
if NEBIUS_MODEL not in available_model_ids:
    qwen_model_ids = sorted(
        model_id for model_id in available_model_ids
        if "qwen" in model_id.lower()
    )
    choices = ", ".join(qwen_model_ids[:20]) or "No Qwen models were returned"
    raise ValueError(
        f"Nebius model {NEBIUS_MODEL!r} is not available to this API key. "
        f"Available Qwen models: {choices}. Set NEBIUS_MODEL to one of those IDs and rerun this cell."
    )

# LangSmith is the evaluation/debugging UI.
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "customer-support-evals"
LANGSMITH_ENDPOINT = os.environ.get(
    "LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"
).rstrip("/")
os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT

# OpenTelemetry is the standard tracing layer. Send it to the same LangSmith region.
os.environ["LANGSMITH_OTEL_ENABLED"] = "true"
os.environ["OTEL_SERVICE_NAME"] = "customer-support-evals-notebook"
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = f"{LANGSMITH_ENDPOINT}/otel"
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = f"{LANGSMITH_ENDPOINT}/otel/v1/traces"
os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"

otel_headers = f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={os.environ['LANGSMITH_PROJECT']}"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = otel_headers
os.environ["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = otel_headers

# LangSmith caches environment lookups in long-lived notebook kernels.
try:
    from langsmith import utils as langsmith_utils
    langsmith_utils.get_env_var.cache_clear()
except (ImportError, AttributeError):
    pass

print("Traces will appear in LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("LangSmith API endpoint:", LANGSMITH_ENDPOINT)
print("Nebius model:", NEBIUS_MODEL)
print("Nebius endpoint:", NEBIUS_BASE_URL)
print("OpenTelemetry traces endpoint:", os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"])


Traces will appear in LangSmith project: customer-support-evals
LangSmith API endpoint: https://api.smith.langchain.com
Nebius model: Qwen/Qwen3-30B-A3B-Instruct-2507
Nebius endpoint: https://api.tokenfactory.nebius.com/v1/
OpenTelemetry traces endpoint: https://api.smith.langchain.com/otel/v1/traces


## 3. Define the ticket categories

We keep the label space small and unambiguous on purpose — when there are 50 categories, *everything* looks like a model failure. Five is a good teaching size.

| Category | What belongs here |
|---|---|
| `order_status` | "Where is my order?", tracking, delivery ETA |
| `refund_request` | Customer wants money back, return-for-refund |
| `product_issue` | Item arrived broken, wrong, defective, or not as described |
| `account_help` | Login, password, address, payment method changes |
| `other` | Anything that doesn't fit above (general questions, feedback) |

In [3]:
CATEGORIES = [
    "order_status",
    "refund_request",
    "product_issue",
    "account_help",
    "other",
]

## 4. Synthetically generate the ticket dataset

We hand-write a small seed set of tickets with **ground-truth labels**, then ask the LLM to generate variations in the same style. This gives us a dataset that:
- has trustworthy labels (we wrote the seeds),
- contains realistic phrasing (the LLM paraphrases),
- includes some intentionally tricky/ambiguous cases (so the agent has something to fail on).

If you want a bigger or smaller dataset, change `EXPANSIONS_PER_SEED` below.

In [4]:
from langchain_nebius import ChatNebius
from pydantic import BaseModel, Field
from typing import List

# Seed tickets — each one is a (message, true_category) pair we trust.
# Many of these are intentionally tricky: they mention keywords from other categories,
# use sarcasm, bury the real intent inside emotional language, or sit on a real edge case.
# The goal is to give the model something to actually fail on.
SEED_TICKETS = [
    # ---------- order_status ----------
    ("Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?", "order_status"),
    ("I never received my order and I want my money back.", "order_status"),  # mentions refund but the real issue is non-delivery
    ("Order shows delivered last Tuesday but it's not at my door, my neighbor's, or the mailroom. What now?", "order_status"),
    ("The website said 2-day shipping. It's been 9 days. Are you kidding me?", "order_status"),  # sarcastic
    ("Tracking link in the email just spins forever. Order #88421.", "order_status"),  # easy to mis-call account_help
    # ---------- refund_request ----------
    ("I'd like to return the headphones I bought last week and get my money back. They're unopened.", "refund_request"),
    ("Where is my refund? I returned the item two weeks ago and still nothing on my card.", "refund_request"),
    ("Please cancel order 99021 and refund my card. I no longer need it.", "refund_request"),  # could feel like account_help
    ("I was charged $89 but the website showed $79 at checkout. Please refund the difference.", "refund_request"),
    ("Returning these for store credit is fine but honestly I'd prefer cash back to my original card.", "refund_request"),
    # ---------- product_issue ----------
    ("The coffee maker arrived with a cracked carafe. Really disappointed.", "product_issue"),
    ("My laptop arrived damaged and I want a full refund, not a replacement.", "product_issue"),  # asks for refund, but root cause is damage
    ("You sent me a size medium shirt but I ordered a large. Second time this has happened.", "product_issue"),
    ("The product description said 'wireless' but I had to buy a separate dongle to use it. Misleading.", "product_issue"),  # subtle "not as described"
    ("Item itself works fine but the box was crushed and the instruction manual is missing.", "product_issue"),
    # ---------- account_help ----------
    ("I can't log into my account — it keeps saying my password is wrong even after I reset it.", "account_help"),
    ("How do I update the credit card on file? I don't see the option anywhere in settings.", "account_help"),
    ("The website won't let me check out — it keeps logging me out mid-payment.", "account_help"),  # sounds like a site bug, technically account
    ("Please remove my old shipping address. I moved last month and don't want stuff going there.", "account_help"),
    ("I keep getting 2FA codes I didn't request. Is someone trying to access my account?", "account_help"),
    # ---------- other ----------
    ("Do you guys ship to Canada? Couldn't find it on the FAQ page.", "other"),
    ("Just wanted to say the customer service rep I spoke to yesterday was amazing. Thank you!", "other"),
    ("Is the red version of SKU-1140 back in stock?", "other"),
    ("Do you offer a student discount? Couldn't find one at checkout.", "other"),
    ("When's your next sale? My birthday is coming up and I'd love to splurge.", "other"),
]

EXPANSIONS_PER_SEED = 3  # 25 seeds * (1 + 3) = 100 tickets total

generator_llm = ChatNebius(
    model=NEBIUS_MODEL,
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url=NEBIUS_BASE_URL,
    temperature=0.9,
)

class Paraphrases(BaseModel):
    variations: List[str] = Field(description="Realistic paraphrases of the original ticket")

def expand(seed_text: str, n: int) -> List[str]:
    if n <= 0:
        return []
    structured = generator_llm.with_structured_output(Paraphrases, method="function_calling")
    out = structured.invoke(
        f"You write realistic e-commerce customer-support tickets.\n"
        f"Produce {n} short paraphrases of the ticket below. KEEP THE SAME UNDERLYING INTENT, "
        f"but vary tone (frustrated, polite, terse, rambling, sarcastic), names, order numbers, "
        f"products, and the specific phrasing. If the original is ambiguous or mentions keywords "
        f"from multiple categories, preserve that ambiguity — don't sanitize it.\n\n"
        f"Original ticket:\n{seed_text}"
    )
    return out.variations

tickets = []
for text, label in SEED_TICKETS:
    tickets.append({"id": f"t{len(tickets):03d}", "text": text, "true_category": label, "source": "seed"})
    for variation in expand(text, EXPANSIONS_PER_SEED):
        tickets.append({"id": f"t{len(tickets):03d}", "text": variation, "true_category": label, "source": "synthetic"})

print(f"Generated {len(tickets)} tickets across {len(CATEGORIES)} categories.")
from collections import Counter
for cat, n in Counter(t["true_category"] for t in tickets).items():
    print(f"  {cat:>15}: {n}")
print()
for t in tickets[:5]:
    print(f"  [{t['true_category']:>15}] {t['text'][:90]}")

Generated 100 tickets across 5 categories.
     order_status: 20
   refund_request: 20
    product_issue: 20
     account_help: 20
            other: 20

  [   order_status] Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me w
  [   order_status] Hey, my order #BLN-8821 was supposed to arrive by now, but the tracking still says 'Proces
  [   order_status] Hi, I placed an order for a kitchen blender (order #BLN-8821) five days ago and haven't se
  [   order_status] Ugh, I ordered a blender five days ago—tracking hasn't moved an inch. Order #BLN-8821. Eit
  [   order_status] I never received my order and I want my money back.


## 5a. What OpenTelemetry adds here

LangSmith is the place where we inspect AI traces and compare baseline vs improved runs. OpenTelemetry is the standard way we emit structured traces from code.

In this notebook, we add one OpenTelemetry parent span around each ticket classification. The LangGraph / LangChain internals still create child spans for the model call, and the parent span carries eval metadata:

- `eval.example_id`: ticket id
- `eval.run_name`: baseline or improved
- `eval.prompt_version`: v1 or v2
- `eval.true_category`: ground-truth label
- `eval.predicted_category`: model output
- `eval.correct`: whether the prediction matched the label
- `eval.reasoning`: model explanation

This makes each row in the CSV traceable back to a specific LangSmith/OpenTelemetry run.

If you see `Failed to export span batch code: 404`, the classifier is still running, but the OTLP exporter is pointed at the wrong URL. This notebook clears stale `OTEL_EXPORTER_*` values and explicitly sends trace spans to `https://api.smith.langchain.com/otel/v1/traces`.


## 5. Build the LangGraph classification agent

LangGraph models an agent as a **graph of nodes**. For a classifier, the graph is tiny — one node that calls the LLM with a structured output schema. We're using LangGraph here (instead of just calling the LLM directly) so the pattern scales to multi-step agents later (e.g., add a retrieval node, a tool-calling node, a confidence-check node).

Because LangSmith tracing is on, **every graph invocation becomes a clickable trace** showing each node's input/output.

In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_nebius import ChatNebius

# --- THIS PROMPT IS WHAT YOU'LL TWEAK LATER ---
CLASSIFIER_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

Return only the category key.

Ticket:
{ticket_text}
"""

class Classification(BaseModel):
    category: Literal["order_status", "refund_request", "product_issue", "account_help", "other"]
    reasoning: str = Field(description="One short sentence explaining the choice.")

class AgentState(TypedDict):
    ticket_text: str
    category: str
    reasoning: str

def build_agent(prompt_template: str):
    """Compile a LangGraph agent. Re-call this any time you change the prompt."""
    llm = ChatNebius(
        model=NEBIUS_MODEL,
        api_key=os.environ["NEBIUS_API_KEY"],
        base_url=NEBIUS_BASE_URL,
        temperature=0,
    ).with_structured_output(Classification, method="function_calling")
    prompt = ChatPromptTemplate.from_template(prompt_template)

    def classify_node(state: AgentState) -> AgentState:
        result = (prompt | llm).invoke({"ticket_text": state["ticket_text"]})
        return {"ticket_text": state["ticket_text"], "category": result.category, "reasoning": result.reasoning}

    graph = StateGraph(AgentState)
    graph.add_node("classify", classify_node)
    graph.add_edge(START, "classify")
    graph.add_edge("classify", END)
    return graph.compile()

agent = build_agent(CLASSIFIER_PROMPT)

# Smoke test on one ticket.
sample = agent.invoke({"ticket_text": tickets[0]["text"], "category": "", "reasoning": ""})
print("Ticket:    ", tickets[0]["text"])
print("Predicted: ", sample["category"])
print("Reasoning: ", sample["reasoning"])

Ticket:     Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?
Predicted:  order_status
Reasoning:  The customer is inquiring about the current status and location of their order, which falls under order status.


In [6]:
from opentelemetry import trace

# This tracer creates ticket-level spans. LangSmith receives them because
# LANGSMITH_OTEL_ENABLED=true is set above.
tracer = trace.get_tracer("customer-support-evals")


## 6. Run the agent on every ticket

Each invocation is automatically traced in LangSmith. After this cell finishes, go to your [LangSmith dashboard](https://smith.langchain.com) → project **`customer-support-evals`** → and you'll see every prediction with full input/output/latency.

In [7]:
import pandas as pd
from tqdm import tqdm

def run_predictions(agent, tickets, run_name="baseline", prompt_version="v1") -> pd.DataFrame:
    rows = []
    for t in tqdm(tickets, desc=f"Classifying {run_name}"):
        # One parent span per ticket makes the spreadsheet row traceable in LangSmith.
        with tracer.start_as_current_span("customer_support.classify_ticket") as span:
            span.set_attribute("langsmith.span.kind", "chain")
            span.set_attribute("eval.run_name", run_name)
            span.set_attribute("eval.prompt_version", prompt_version)
            span.set_attribute("eval.example_id", t["id"])
            span.set_attribute("eval.true_category", t["true_category"])
            span.set_attribute("input.ticket_text", t["text"])

            out = agent.invoke({"ticket_text": t["text"], "category": "", "reasoning": ""})
            correct = t["true_category"] == out["category"]

            span.set_attribute("eval.predicted_category", out["category"])
            span.set_attribute("eval.correct", correct)
            span.set_attribute("eval.reasoning", out["reasoning"])
            span.set_attribute("output.category", out["category"])

            rows.append({
                "id": t["id"],
                "ticket_text": t["text"],
                "true_category": t["true_category"],
                "predicted_category": out["category"],
                "reasoning": out["reasoning"],
                "correct": correct,
                "otel_run_name": run_name,
                "otel_prompt_version": prompt_version,
            })
    return pd.DataFrame(rows)

results_v1 = run_predictions(agent, tickets, run_name="baseline", prompt_version="v1")
results_v1.head(10)


Classifying baseline: 100%|██████████| 100/100 [01:27<00:00,  1.15it/s]


,id,ticket_text,true_category,predicted_category,reasoning,correct,otel_run_name,otel_prompt_version
0,t000,"Hi, I ordered a blender 5 days ago and the tra...",order_status,order_status,The customer is inquiring about the current st...,True,baseline,v1
1,t001,"Hey, my order #BLN-8821 was supposed to arrive...",order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
2,t002,"Hi, I placed an order for a kitchen blender (o...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
3,t003,"Ugh, I ordered a blender five days ago—trackin...",order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
4,t004,I never received my order and I want my money ...,order_status,refund_request,The customer explicitly states they did not re...,False,baseline,v1
5,t005,"Hey, my order #8821 was supposed to arrive las...",order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
6,t006,"Hi, I'm writing to report that I haven't gotte...",order_status,refund_request,The customer is requesting a refund due to not...,False,baseline,v1
7,t007,"So, I ordered the blue wireless headphones (or...",order_status,refund_request,The customer is asking for a refund due to a d...,False,baseline,v1
8,t008,Order shows delivered last Tuesday but it's no...,order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
9,t009,"My order was marked as delivered last Tuesday,...",order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1


## 7. Evaluate: accuracy, precision, recall

- **Accuracy** = fraction of tickets classified correctly overall.
- **Precision (per class)** = of all the tickets the model *called* `refund_request`, how many actually were? High precision = few false alarms.
- **Recall (per class)** = of all the tickets that *truly were* `refund_request`, how many did the model catch? High recall = few misses.

**Why look at both:** a model that *always* predicts `other` will have 20% accuracy and 100% recall on `other` but 0% recall on everything else. Per-class precision/recall exposes that immediately.

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate(df: pd.DataFrame, label: str):
    y_true = df["true_category"]
    y_pred = df["predicted_category"]
    print(f"=== {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}  ({df['correct'].sum()}/{len(df)} correct)\n")
    print("Per-class precision / recall / F1:")
    print(classification_report(y_true, y_pred, labels=CATEGORIES, zero_division=0))
    print("Confusion matrix (rows = true, cols = predicted):")
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=CATEGORIES, columns=CATEGORIES,
    )
    print(cm)
    return accuracy_score(y_true, y_pred)

acc_v1 = evaluate(results_v1, "Run 1 — baseline prompt")

=== Run 1 — baseline prompt ===
Accuracy: 90.00%  (90/100 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       0.85      0.85      0.85        20
refund_request       0.74      1.00      0.85        20
 product_issue       1.00      0.80      0.89        20
  account_help       1.00      1.00      1.00        20
         other       1.00      0.85      0.92        20

      accuracy                           0.90       100
     macro avg       0.92      0.90      0.90       100
  weighted avg       0.92      0.90      0.90       100

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              17               3              0             0   
refund_request             0              20              0             0   
product_issue              0               4             16             0   
account_help               0    

## 8. Export to spreadsheet for validator comments

This is the **human-in-the-loop** step. Open `results_v1.csv` in Google Sheets or Excel.

Each row has an empty `validator_comment` column. Your job:

1. **Filter `correct == FALSE`** to see only the failures.
2. For each failure, write a short note in `validator_comment` — e.g. *"complaint about delivery, model called it product_issue"*, *"ambiguous, I'd accept either"*, *"label is wrong, this really is `other`"*.
3. Also scan a sample of `correct == TRUE` rows — sometimes the model gets the right label for the *wrong reason*.
4. Once you've annotated, **cluster the comments into 2–4 failure categories** in a separate tab. For example:
   - *"Confuses `order_status` with `refund_request` when the customer mentions both delivery and money"*
   - *"Calls polite thank-you messages `account_help`"*
   - *"Routes 'wrong item' as `refund_request` instead of `product_issue`"*
5. **Pick the one failure category that matters most for your use case** (the one that's most expensive if it happens in production), and bring it back to step 9.

In [9]:
results_v1_export = results_v1.copy()
results_v1_export["validator_comment"] = ""
results_v1_export["failure_category"] = ""
results_v1_export.to_csv("results_v1.csv", index=False)
print("Wrote results_v1.csv — open it in Google Sheets or Excel.")
print("Columns:", list(results_v1_export.columns))

Wrote results_v1.csv — open it in Google Sheets or Excel.
Columns: ['id', 'ticket_text', 'true_category', 'predicted_category', 'reasoning', 'correct', 'otel_run_name', 'otel_prompt_version', 'validator_comment', 'failure_category']


## 9. Tweak the prompt to fix the failure category you picked

Edit `IMPROVED_PROMPT` below to address the failure you chose. Some common moves:

- **Add disambiguation rules.** *"If the customer mentions both delivery delay AND a refund request, classify as `order_status` — the refund is downstream of the delivery problem."*
- **Add a few-shot example** of the exact failure case with the correct label.
- **Tighten a category definition.** *"`account_help` is ONLY for login/password/profile issues, not for general site bugs."*
- **Force a step.** *"First identify the customer's primary intent in one sentence, then pick the category."*

Keep the change focused on the **one failure category** you picked. If you change everything, you won't know what helped.

In [10]:
IMPROVED_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA, or non-delivery
- refund_request: the customer is asking for their money back (and the item itself is fine, or already returned)
- product_issue: the item arrived broken, wrong, defective, or not as described — even if the customer also asks for a refund as the remedy
- account_help: login, password, address, or payment method changes
- other: general questions, feedback, browsing, anything not covered above

Disambiguation rules:
1. If a damaged/wrong/defective item is mentioned, the category is product_issue — regardless of what remedy the customer asks for.
2. If the customer never received the order, the category is order_status — even if they mention wanting their money back.
3. If the customer is asking about a refund they already initiated ("where is my refund?"), the category is refund_request.
4. Site bugs that prevent checkout are account_help.

Think step by step:
1. What is the customer's *primary* problem? (one sentence)
2. Which category fits that problem best?

Ticket:
{ticket_text}
"""

agent_v2 = build_agent(IMPROVED_PROMPT)
results_v2 = run_predictions(agent_v2, tickets, run_name="improved", prompt_version="v2")

results_v2_export = results_v2.copy()
results_v2_export["validator_comment"] = ""
results_v2_export["failure_category"] = ""
results_v2_export.to_csv("results_v2.csv", index=False)

acc_v2 = evaluate(results_v2, "Run 2 — improved prompt")

Classifying improved: 100%|██████████| 100/100 [01:41<00:00,  1.02s/it]

=== Run 2 — improved prompt ===
Accuracy: 99.00%  (99/100 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       1.00      1.00      1.00        20
refund_request       1.00      0.95      0.97        20
 product_issue       0.95      1.00      0.98        20
  account_help       1.00      1.00      1.00        20
         other       1.00      1.00      1.00        20

      accuracy                           0.99       100
     macro avg       0.99      0.99      0.99       100
  weighted avg       0.99      0.99      0.99       100

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              20               0              0             0   
refund_request             0              19              1             0   
product_issue              0               0             20             0   
account_help               0    

## 10. Compare the two runs

Now look at the headline accuracy and at *which specific tickets flipped* between runs. Some will go from wrong → right (the win you were aiming for). Some may go from right → wrong (a regression you caused). This is normal — almost no prompt change is strictly Pareto-better, and the trade-offs are the most important thing to understand.

In [11]:
print(f"Accuracy v1: {acc_v1:.2%}")
print(f"Accuracy v2: {acc_v2:.2%}")
print(f"Δ          : {(acc_v2 - acc_v1):+.2%}\n")

comparison = results_v1.merge(
    results_v2[["id", "predicted_category", "reasoning", "correct"]],
    on="id", suffixes=("_v1", "_v2"),
)

flipped = comparison[comparison["predicted_category_v1"] != comparison["predicted_category_v2"]]
print(f"{len(flipped)} tickets changed prediction between runs.\n")

wins   = flipped[(~flipped["correct_v1"]) & (flipped["correct_v2"])]
losses = flipped[(flipped["correct_v1"]) & (~flipped["correct_v2"])]
print(f"Wins (wrong → right):     {len(wins)}")
print(f"Regressions (right → wrong): {len(losses)}")

flipped[["ticket_text", "true_category", "predicted_category_v1", "predicted_category_v2"]]

Accuracy v1: 90.00%
Accuracy v2: 99.00%
Δ          : +9.00%

11 tickets changed prediction between runs.

Wins (wrong → right):     10
Regressions (right → wrong): 1


,ticket_text,true_category,predicted_category_v1,predicted_category_v2
4,I never received my order and I want my money ...,order_status,refund_request,order_status
6,"Hi, I'm writing to report that I haven't gotte...",order_status,refund_request,order_status
7,"So, I ordered the blue wireless headphones (or...",order_status,refund_request,order_status
21,"Hey, I'm really disappointed—these headphones ...",refund_request,refund_request,product_issue
44,My laptop arrived damaged and I want a full re...,product_issue,refund_request,product_issue
45,"Ugh, the laptop I ordered (order #88219) came ...",product_issue,refund_request,product_issue
46,"Dear Support, my order #88219 arrived with vis...",product_issue,refund_request,product_issue
47,"Hey, just got my laptop (order #88219) and it’...",product_issue,refund_request,product_issue
89,"Hey, I've been checking for weeks now—any upda...",other,order_status,other
90,Could you please confirm if the red variant of...,other,order_status,other


## 11. What to take away

- **LangGraph** gave us a clean shape for the agent. Right now it's one node, but you can drop in retrieval, tools, or a self-check node without rewriting the eval harness.
- **LangSmith** turned every LLM call into an inspectable trace. OpenTelemetry added a standard parent span around each ticket so CSV rows, prompt versions, labels, and correctness are attached to the trace.
- **Accuracy alone is a trap.** Per-class precision/recall and the confusion matrix tell you *what kind* of mistakes the model is making.
- **Validator comments are the most valuable artifact in this whole notebook.** Numbers tell you *that* something is wrong; human notes tell you *what* and *why*.
- **Prompt iteration is a measure → diagnose → fix → re-measure loop.** Without the dataset and the metrics, prompt tweaks are just vibes.

### Suggested next steps
- Replace the synthetic seed tickets with anonymized **real** tickets from your inbox.
- Upload the dataset to LangSmith as a versioned **Dataset** and use the LangSmith `evaluate()` runner so each prompt version gets a saved score.
- Add a second node to the graph — e.g., a confidence check that routes low-confidence predictions to a human queue.